#**Aula 1 - Desafios - Desafio 2**

## **Caso de uso - Marketplace de classificados veículos**

###Desafio 2 - EXTRA:

#####2.1 - Crie um processo (função) para analisar se o item Infringe alguma regra de publicação do Marketiplace.

```
Por exemplo: Se o item for um veículo especial (ambulância, funerário ou guindaste),
ele não é permitido na plataforma e a analise deve sinalizar que esse item é suspeito.
```

  - Analise os casos de forma independente, pois pode aumentar a efetividade do modelo.
  - Ajuste seu prompt para incluir uma coluna de justificativa, ou seja, peça para o LLM justificar a decisão e inclua na análise final.
  - Utilize a técnica para transformar o dataframe em CSV (texto) no prompt.
  - Estrure a resposta do prompt para um formato JSON, com as tags: flg_suspeito_genai e comentario_genai,
  - Salve seu resultado no dataframe inicial para comparar o resultado.
  - Compare seu resultado com a marcação de item suspeito (flg_suspeito).  
  - Para evitar custos desnecessários, uma dica seria fazer um sample(5, random_state=42) no dataframe.

**Dica:**
- Verifique se todos os dados (colunas) serão necessários para a análise.

In [1]:
OPENAI_API_KEY = "" # @param {"type":"string"}

In [2]:
# Conexão com a API da OpenAI
from openai import OpenAI

client = OpenAI(
    api_key = OPENAI_API_KEY
)


In [3]:
import pandas as pd

# URL para exportação CSV (ajustada com export?format=csv)
url = "https://docs.google.com/spreadsheets/d/1-ISE6Z3ra8dKnpqe2GLh1oCxnLcp5i0d1Izd8iQpA1Y/export?format=csv&gid=0"

# Carregar no DataFrame
df_base_itens = pd.read_csv(url)

# Visualizar
df_base_itens.head()

,id_item,id_seller,item_date_create,user_type_description,title_item,description_item,detail_item,brand_item,model_item,price_item,year_item,year_model_item,state_item,address_item,flg_suspeito
0,IT0027,SL0092,26/01/2025,não concessionário,Ciclomotor Elétrico 50cc,"Ciclomotor elétrico, ideal para cidade. Super ...",ciclomotor,Shineray,PT-2,7200,2024,2024,BA,"Av. Tancredo Neves, 300, Salvador, 41820-021",1
1,IT0028,SL0133,28/01/2025,concessionário,Vendo Ambulância Renault Master,"Ambulância usada, tipo A, com todos os equipam...",furgão,Renault,Master,115000,2018,2018,MG,"Av. do Contorno, 1500, Belo Horizonte, 30110-005",1
2,IT0018,SL1055,18/01/2025,não concessionário,Kart de Competição Thunder,"Kart para competição, motor preparado, pronto ...",motoneta,Yamaha,NMAX 160,28000,2023,2023,SP,"Av. Interlagos, 500, São Paulo, 04777-000",1
3,IT0009,SL1122,10/01/2025,concessionário,Fiat Strada Freedom 1.3 2022,Picape versátil e econômica. Versão cabine plu...,picape,Fiat,Strada,98000,2022,2022,SP,"Av. Ibirapuera, 3000, São Paulo, 04028-002",0
4,IT0014,SL1212,15/01/2025,concessionário,Volvo FH 540 Globetrotter 6x4 2023,Cavalo mecânico top de linha. Pacote de segura...,caminhão,Volvo,FH 540,950000,2023,2023,PR,"BR-116, 15000, Curitiba, 81690-100",0


In [4]:
df_base_itens.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30 entries, 0 to 29
Data columns (total 15 columns):
 #   Column                 Non-Null Count  Dtype 
---  ------                 --------------  ----- 
 0   id_item                30 non-null     object
 1   id_seller              30 non-null     object
 2   item_date_create       30 non-null     object
 3   user_type_description  30 non-null     object
 4   title_item             30 non-null     object
 5   description_item       30 non-null     object
 6   detail_item            30 non-null     object
 7   brand_item             30 non-null     object
 8   model_item             30 non-null     object
 9   price_item             30 non-null     int64 
 10  year_item              30 non-null     int64 
 11  year_model_item        30 non-null     int64 
 12  state_item             30 non-null     object
 13  address_item           30 non-null     object
 14  flg_suspeito           30 non-null     int64 
dtypes: int64(4), object(11)
m

In [5]:
df_base_itens.head()

,id_item,id_seller,item_date_create,user_type_description,title_item,description_item,detail_item,brand_item,model_item,price_item,year_item,year_model_item,state_item,address_item,flg_suspeito
0,IT0027,SL0092,26/01/2025,não concessionário,Ciclomotor Elétrico 50cc,"Ciclomotor elétrico, ideal para cidade. Super ...",ciclomotor,Shineray,PT-2,7200,2024,2024,BA,"Av. Tancredo Neves, 300, Salvador, 41820-021",1
1,IT0028,SL0133,28/01/2025,concessionário,Vendo Ambulância Renault Master,"Ambulância usada, tipo A, com todos os equipam...",furgão,Renault,Master,115000,2018,2018,MG,"Av. do Contorno, 1500, Belo Horizonte, 30110-005",1
2,IT0018,SL1055,18/01/2025,não concessionário,Kart de Competição Thunder,"Kart para competição, motor preparado, pronto ...",motoneta,Yamaha,NMAX 160,28000,2023,2023,SP,"Av. Interlagos, 500, São Paulo, 04777-000",1
3,IT0009,SL1122,10/01/2025,concessionário,Fiat Strada Freedom 1.3 2022,Picape versátil e econômica. Versão cabine plu...,picape,Fiat,Strada,98000,2022,2022,SP,"Av. Ibirapuera, 3000, São Paulo, 04028-002",0
4,IT0014,SL1212,15/01/2025,concessionário,Volvo FH 540 Globetrotter 6x4 2023,Cavalo mecânico top de linha. Pacote de segura...,caminhão,Volvo,FH 540,950000,2023,2023,PR,"BR-116, 15000, Curitiba, 81690-100",0


In [6]:
# Selecionando uma amostra aleatória do dataframe
df_base_itens_anl = df_base_itens.sample(1, random_state=42).drop(columns=['flg_suspeito'])
df_base_itens_anl

,id_item,id_seller,item_date_create,user_type_description,title_item,description_item,detail_item,brand_item,model_item,price_item,year_item,year_model_item,state_item,address_item
27,IT0015,SL9831,15/01/2025,não concessionário,Vendo Trator Valtra A950,"Trator agrícola em perfeito estado, ótimo para...",caminhão,Scania,R450,310000,2021,2021,MT,"Rua dos Colonizadores, 150, Sinop, 78550-000"


In [7]:
dados_exemplo = df_base_itens_anl.to_csv(index=False, sep='|')
print(dados_exemplo)

id_item|id_seller|item_date_create|user_type_description|title_item|description_item|detail_item|brand_item|model_item|price_item|year_item|year_model_item|state_item|address_item
IT0015|SL9831|15/01/2025|não concessionário|Vendo Trator Valtra A950|Trator agrícola em perfeito estado, ótimo para lavoura. Poucas horas de uso. Tratar com João.|caminhão|Scania|R450|310000|2021|2021|MT|Rua dos Colonizadores, 150, Sinop, 78550-000



[PDF - Quem Somos e Poítica de Uso](https://drive.google.com/file/d/1-ZpUOl8OA4lxa8CJ6auT42hSxaF3jclk/view?usp=sharing)

In [8]:
# Estruturação do prompt
prompt = [
    {
            "role": "developer",
            "content": [
                {
                  "type": "input_text",
                  "text": """
                      Você é um especialista em segurança de dados e é respossável por fazer análise dos dados de
                      publicação de veículos no marketplace com base nas politicas de publização para vendedores (sellers).

                      # Trecho das politicas de uso do marketplace
                      3. Políticas para Vendedores (Sellers)
                        3.1 Veículos permitidos
                          - Passageiros: Automóveis (sedan, hatch, SUV, picape, minivan), van, furgão, micro-ônibus, ônibus, motonetas, motocicletas, triciclos, quadriciclos.
                          - Carga: Utilitários, caminhões, camionetas.
                          - Motorização: Combustão, elétricos e híbridos.
                        3.2 Veículos proibidos
                          - Veículos especiais (ambulâncias, funerários, guindastes etc.).
                          - Veículos de competição (carros de corrida, motocross, harts etc.).
                          - Tratores, bicicletas, reboques, semirreboques, charretes, carroças, bondes, ciclomotores*.

                            *Ciclomotores: duas ou três rodas, até 50 cm³ ou equivalente elétrico, e velocidade máxima de até 50 km/h.
                        3.3 Regras de publicação
                        - É proibido incluir nas fotos ou descrições: números de telefone, celular, WhatsApp, e-mails ou endereços de localização (logradouros).
                        - O contato entre comprador e vendedor só é liberado pela plataforma após solicitação de interesse (lead) pelo comprador.
                        - Informações com a marca, logotipo, nome do vendedor e nome da concessionária é permitido.

                      Analise os dados seguir e valide se possuem algum tipo de violação de regra da plataforma.
                      Responda no formato JSON solicitado e deve conter as tags 'flg_suspeito' e 'comentario' para cada registro.

                      EXEMPLO DE RESPOSTA:
                      {"flg_suspeito":"suspeito/não suspeito", "comentario": "comentário resumido sobre o motivo da suspeita, se não houver suspeita colcoar 'sem suspeita'."}

                      Não adicione explicações extras, responda **apenas** com o JSON.
                    """
                  }
              ]
            },
         {
             "role": "user",
             "content": [
                 {
                     "type": "input_text",
                     "text": f"**Dados do veículo:** {dados_exemplo}"},
        ],
    }]

prompt

[{'role': 'developer',
  'content': [{'type': 'input_text',
    'text': '\n                      Você é um especialista em segurança de dados e é respossável por fazer análise dos dados de\n                      publicação de veículos no marketplace com base nas politicas de publização para vendedores (sellers).\n\n                      # Trecho das politicas de uso do marketplace\n                      3. Políticas para Vendedores (Sellers)\n                        3.1 Veículos permitidos\n                          - Passageiros: Automóveis (sedan, hatch, SUV, picape, minivan), van, furgão, micro-ônibus, ônibus, motonetas, motocicletas, triciclos, quadriciclos.\n                          - Carga: Utilitários, caminhões, camionetas.\n                          - Motorização: Combustão, elétricos e híbridos.\n                        3.2 Veículos proibidos\n                          - Veículos especiais (ambulâncias, funerários, guindastes etc.).\n                          - Veículos de c

In [9]:
from openai import OpenAI

client = OpenAI(
    api_key=OPENAI_API_KEY
)

response = client.responses.create(
    model="gpt-4.1-mini",
    temperature=0.7,
    input=prompt
)

print(response.output_text)

{"flg_suspeito":"suspeito", "comentario":"Veículo proibido (trator) e endereço completo incluído na descrição."}


In [10]:
# Converte string para json
import json

response_json = json.loads(response.output_text)
type(response_json)

dict

In [11]:
response_json

{'flg_suspeito': 'suspeito',
 'comentario': 'Veículo proibido (trator) e endereço completo incluído na descrição.'}

---

**Obs.:** Não tem endereço na descrição, certo?
Temos que revisar os dados que estamos enviando para o prompt.

---

In [12]:
# Seleciona apenas colunas importantes e relevantes para a análise
df_base_itens_anl = df_base_itens.sample(1, random_state=42).drop(columns=['flg_suspeito','item_date_create','user_type_description','year_item','year_model_item', 'state_item','address_item'])
df_base_itens_anl

,id_item,id_seller,title_item,description_item,detail_item,brand_item,model_item,price_item
27,IT0015,SL9831,Vendo Trator Valtra A950,"Trator agrícola em perfeito estado, ótimo para...",caminhão,Scania,R450,310000


Criando uma função

In [13]:
import time

# Criando a função
def fn_analisa_dados(df_dados):
  """ Função que analisa se os dados um veículo estão seguindo ou não as regras de uso da plataforma.
  """
  # Log
  print("Iniciando a análise do seller:")
  print(df_dados.id_seller)
  #print('df_dados')
  #print(df_dados)

  # Cria CSV apenas com o cabeçalho e esta linha
  csv_dados = df_dados.to_frame().T.to_csv(index=False, sep='|')
  #print('csv_dados')
  #print(csv_dados)

  prompt = [
    {
            "role": "developer",
            "content": [
                {
                  "type": "input_text",
                  "text": """
                      Você é um especialista em segurança de dados e é respossável por fazer análise dos dados de
                      publicação de veículos no marketplace com base nas politicas de publização para vendedores (sellers).

                      # Trecho das politicas de uso do marketplace
                      3. Políticas para Vendedores (Sellers)
                        3.1 Veículos permitidos
                          - Passageiros: Automóveis (sedan, hatch, SUV, picape, minivan), van, furgão, micro-ônibus, ônibus, motonetas, motocicletas, triciclos, quadriciclos.
                          - Carga: Utilitários, caminhões, camionetas.
                          - Motorização: Combustão, elétricos e híbridos.
                        3.2 Veículos proibidos
                          - Veículos especiais (ambulâncias, funerários, guindastes etc.).
                          - Veículos de competição (carros de corrida, motocross, harts etc.).
                          - Tratores, bicicletas, reboques, semirreboques, charretes, carroças, bondes, ciclomotores*.

                            *Ciclomotores: duas ou três rodas, até 50 cm³ ou equivalente elétrico, e velocidade máxima de até 50 km/h.
                        3.3 Regras de publicação
                        - É proibido incluir nas fotos ou descrições: números de telefone, celular, WhatsApp, e-mails ou endereços de localização (logradouros).
                        - O contato entre comprador e vendedor só é liberado pela plataforma após solicitação de interesse (lead) pelo comprador.
                        - Informações com a marca, logotipo, nome do vendedor e nome da concessionária é permitido.

                      Analise os dados seguir e valide se possuem algum tipo de violação de regra da plataforma.
                      Responda no formato JSON solicitado e deve conter as tags 'flg_suspeito' e 'comentario' para cada registro.

                      EXEMPLO DE RESPOSTA:
                      {"flg_suspeito":"suspeito/não suspeito", "comentario": "comentário resumido sobre o motivo da suspeita, se não houver suspeita colcoar 'sem suspeita'."}

                      Não adicione explicações extras, responda **apenas** com o JSON.
                    """
                  }
              ]
            },
         {
             "role": "user",
             "content": [
                 {
                     "type": "input_text",
                     "text": f"**Dados do veículo:** {csv_dados}"},
        ],
    }]

  response = client.responses.create(
    model="gpt-4.1-mini",
    temperature=0.7,
    input=prompt
  )

  response_json = json.loads(response.output_text)
  #print('response_json')
  #print(response_json)

  # Log
  time.sleep(2)  # Pausa por alguns segundos
  print("Finalizando análise!\n")

  return response_json['flg_suspeito'], response_json['comentario']

In [14]:
# Teste da função com um caso
df_base_itens_anl[['flg_suspeito_genai','comentario_genai']] = df_base_itens_anl.apply(fn_analisa_dados, axis=1).apply(pd.Series)
df_base_itens_anl

Iniciando a análise do seller:
SL9831
Finalizando análise!



,id_item,id_seller,title_item,description_item,detail_item,brand_item,model_item,price_item,flg_suspeito_genai,comentario_genai
27,IT0015,SL9831,Vendo Trator Valtra A950,"Trator agrícola em perfeito estado, ótimo para...",caminhão,Scania,R450,310000,suspeito,Veículo proibido (trator) listado no marketplace.


Agora podemos aplicar para todos os dados e validar o resultado, certo?

In [15]:
df_base_itens[['flg_suspeito_genai','comentario_genai']] = df_base_itens.drop(columns=['flg_suspeito','item_date_create','user_type_description','year_item','year_model_item', 'state_item','address_item']).apply(fn_analisa_dados, axis=1).apply(pd.Series)

Iniciando a análise do seller:
SL0092
Finalizando análise!

Iniciando a análise do seller:
SL0133
Finalizando análise!

Iniciando a análise do seller:
SL1055
Finalizando análise!

Iniciando a análise do seller:
SL1122
Finalizando análise!

Iniciando a análise do seller:
SL1212
Finalizando análise!

Iniciando a análise do seller:
SL1234
Finalizando análise!

Iniciando a análise do seller:
SL1234
Finalizando análise!

Iniciando a análise do seller:
SL2109
Finalizando análise!

Iniciando a análise do seller:
SL3344
Finalizando análise!

Iniciando a análise do seller:
SL3344
Finalizando análise!

Iniciando a análise do seller:
SL3434
Finalizando análise!

Iniciando a análise do seller:
SL3456
Finalizando análise!

Iniciando a análise do seller:
SL4321
Finalizando análise!

Iniciando a análise do seller:
SL4815
Finalizando análise!

Iniciando a análise do seller:
SL4815
Finalizando análise!

Iniciando a análise do seller:
SL5566
Finalizando análise!

Iniciando a análise do seller:
SL5678
Fi

In [16]:
df_base_itens

,id_item,id_seller,item_date_create,user_type_description,title_item,description_item,detail_item,brand_item,model_item,price_item,year_item,year_model_item,state_item,address_item,flg_suspeito,flg_suspeito_genai,comentario_genai
0,IT0027,SL0092,26/01/2025,não concessionário,Ciclomotor Elétrico 50cc,"Ciclomotor elétrico, ideal para cidade. Super ...",ciclomotor,Shineray,PT-2,7200,2024,2024,BA,"Av. Tancredo Neves, 300, Salvador, 41820-021",1,suspeito,"Veículo categorizado como ciclomotor, que é pr..."
1,IT0028,SL0133,28/01/2025,concessionário,Vendo Ambulância Renault Master,"Ambulância usada, tipo A, com todos os equipam...",furgão,Renault,Master,115000,2018,2018,MG,"Av. do Contorno, 1500, Belo Horizonte, 30110-005",1,suspeito,Veículo proibido (ambulância) e inclusão de nú...
2,IT0018,SL1055,18/01/2025,não concessionário,Kart de Competição Thunder,"Kart para competição, motor preparado, pronto ...",motoneta,Yamaha,NMAX 160,28000,2023,2023,SP,"Av. Interlagos, 500, São Paulo, 04777-000",1,suspeito,Veículo de competição (kart de competição) pro...
3,IT0009,SL1122,10/01/2025,concessionário,Fiat Strada Freedom 1.3 2022,Picape versátil e econômica. Versão cabine plu...,picape,Fiat,Strada,98000,2022,2022,SP,"Av. Ibirapuera, 3000, São Paulo, 04028-002",0,não suspeito,sem suspeita
4,IT0014,SL1212,15/01/2025,concessionário,Volvo FH 540 Globetrotter 6x4 2023,Cavalo mecânico top de linha. Pacote de segura...,caminhão,Volvo,FH 540,950000,2023,2023,PR,"BR-116, 15000, Curitiba, 81690-100",0,não suspeito,sem suspeita
5,IT0001,SL1234,02/01/2025,concessionário,Renault Kwid Zen 1.0 2023,"Veículo em excelente estado, com baixa quilome...",hatch,Renault,Kwid,58000,2023,2023,SP,"Rua Haddock Lobo, 595, São Paulo, 01414-001",0,não suspeito,sem suspeita
6,IT0017,SL1234,17/01/2025,concessionário,Jeep Compass S 1.3 Turbo 4x4 2024,"SUV top de linha, com teto solar panorâmico, s...",SUV,Jeep,Compass,230000,2024,2024,SP,"Rua Haddock Lobo, 595, São Paulo, 01414-001",0,não suspeito,sem suspeita
7,IT0006,SL2109,07/01/2025,não concessionário,Jeep Renegade Longitude 1.8 2018,"Versão Longitude, flex, automático. Pneus novo...",SUV,Jeep,Renegade,85000,2018,2018,SP,"Rua Augusta, 1500, São Paulo, 01304-001",0,não suspeito,sem suspeita.
8,IT0010,SL3344,11/01/2025,não concessionário,BMW 320i GP 2.0 Turbo 2020,"Sedan premium, impecável. Baixa km. Interior c...",sedan,BMW,320i,215000,2020,2020,RJ,"Rua Visconde de Pirajá, 550, Rio de Janeiro, 2...",0,não suspeito,sem suspeita
9,IT0023,SL3344,21/01/2025,não concessionário,Audi A3 Sedan S-Line 2.0 TFSI 2018,"Carro para pessoas exigentes. Pacote S-Line, t...",sedan,Audi,A3,155000,2018,2018,RJ,"Rua Visconde de Pirajá, 550, Rio de Janeiro, 2...",0,não suspeito,sem suspeita


**🪄 Isso não é mágico??? 🪄**